# LAB 04 — Streaming & Auto Loader

> **Ticket DATA-205** · From: E-commerce · Priority: High
>
> The nightly batch loads orders too late: the shop drops order files into the landing volume during the day, and E-commerce wants them in Bronze soon after they land. A load that runs again must not add duplicate rows. New fields appear in the files without notice, and the load must keep them instead of losing them.

**Definition of done**
- Task 1: `bronze.orders_stream` is loaded with one SQL statement, and a second run of the same statement adds 0 rows.
- Task 2: `df_stream` is an Auto Loader stream over the landing zone of Tasks 2–4.
- Task 3: the stream writes `bronze.lab04_orders_autoloader`, processes every waiting file and stops by itself.
- Task 4: a run with no new files adds 0 rows. After a new file lands, the next run adds exactly the rows of that file.
- Task 5 (stretch): every row of `bronze.orders_with_metadata` has `_processing_time` and `_source_file`.
- Task 6: `bronze.lab04_orders_rescued` keeps the fields outside the declared schema in `_rescued_data`.

A task is done when its validation cell prints `Task N OK`. For Tasks 1 and 3–6, **Check your work** in the app must pass as well. Task 5 is stretch: do it if you finish early, or after the course. Task 6 does not depend on it.

Work in the Workshop Guide app in the left window, and paste the code into the marked cells of this notebook.

**Prerequisites:** 04 — Incremental Ingestion demo

## Setup

> **Run the Setup cells again only after a session restart** (for example after a break). The second cell deletes the table of Task 1 and the checkpoint and schema of Tasks 2–4. After a restart, run both Setup cells, then Task 1, the Task 2 reset cell, Task 2 and Task 3 again, and Task 4 if you had done it.

In [0]:
%run ../../setup/00_setup

In [0]:
# Prepare landing zone and checkpoint paths
landing_path = f"{DATASET_PATH}/orders/stream"
lab_base = f"{DATASET_PATH}/lab04_work"     # checkpoints, schemas and the Auto Loader landing zone (UC Volume)
checkpoint_path = f"{lab_base}/checkpoint"
schema_path = f"{lab_base}/schema"
target_table = f"{CATALOG}.{BRONZE_SCHEMA}.orders_stream"

# Clean up from previous runs
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

print(f"Landing path: {landing_path}")
print(f"Target table: {target_table}")
print(f"Files available: {[f.name for f in dbutils.fs.ls(landing_path)]}")

## Task 1 — Load the Files Once, Then Run the Load Again

Start with the simplest option: one SQL statement that loads the files of the landing zone and skips the files it has already loaded. You use it when a scheduled job loads new files, because the job runs the same statement again and again. A second run must add 0 rows.

The cell creates the target table, runs the load twice and prints the row count after each run. Get the code from the Workshop Guide, **Task 1**.

In [ ]:
# Task 1: paste the code from the Workshop Guide here

In [ ]:
# -- Validation --
assert count_after_copy > 0, "The first run should have loaded data"
assert count_after_rerun == count_after_copy, "The load should be idempotent - no new rows on re-run!"
print(f"Task 1 OK: {count_after_copy} rows loaded; re-run added 0 rows (idempotent)")

## Task 2 — Configure the Auto Loader Stream

Files arrive during the day, so each load should read only the new ones. Auto Loader is a streaming source that finds new files by itself. It saves the inferred schema in a schema location, so every run reads the same columns. Tasks 2–4 use their own landing zone, so in Task 4 you can watch a new file arrive.

1. Run the provided reset cell below without changes. It resets the table, the checkpoint and the landing zone of Tasks 2–4.
2. Build `df_stream` in the cell after it. Get the code from the Workshop Guide, **Task 2**.

**Provided cell: run the next cell without changes.**

In [0]:
# PROVIDED — run as-is (no changes needed)
# Resets the Task 2–4 table, checkpoint and landing zone
al_target = f"{CATALOG}.{BRONZE_SCHEMA}.lab04_orders_autoloader"
spark.sql(f"DROP TABLE IF EXISTS {al_target}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

# Private landing zone for Tasks 2-4: all source files except the last one,
# which "arrives" later in Task 4
al_landing = f"{lab_base}/landing"
dbutils.fs.rm(al_landing, True)
stream_files = sorted(f.path for f in dbutils.fs.ls(landing_path) if f.name.endswith(".json"))
for p in stream_files[:-1]:
    dbutils.fs.cp(p, f"{al_landing}/{p.split('/')[-1]}")
late_file = stream_files[-1]
print(f"Auto Loader landing zone: {al_landing} ({len(stream_files) - 1} files; 1 more arrives in Task 4)")

In [0]:
# Task 2: paste the code from the Workshop Guide here

In [0]:
# -- Validation --
assert df_stream.isStreaming, "Should be a streaming DataFrame"
print(f"Task 2 OK: Streaming DataFrame configured with schema: {df_stream.schema.fieldNames()}")

## Task 3 — Write the Stream and Stop When the Files Are Done

The stream writes to a Delta table in Bronze. It must process every waiting file and then stop by itself, so a scheduled job does not keep compute running all day. The checkpoint records which files are done.

Get the code from the Workshop Guide, **Task 3**. The table gets fewer rows than in Task 1, because the landing zone of Tasks 2–4 holds every source file except the last one. That file arrives in Task 4.

In [0]:
# Task 3: paste the code from the Workshop Guide here

In [0]:
# -- Validation --
al_count = spark.table(al_target).count()
assert al_count > 0, "Auto Loader should have loaded data"
print(f"Task 3 OK: {al_count} rows loaded via Auto Loader")

## Task 4 — Run the Ingestion Before and After a New File

A scheduled job runs the same code every time. Put the read of Task 2 and the write of Task 3 into `run_ingestion()` and check that the checkpoint works in both cases. A run with no new files adds **0 rows**. A run after a new file lands adds **exactly the rows of that file**.

The cell runs Run A (no new files), copies a late file into the landing zone and then runs Run B. Get the code from the Workshop Guide, **Task 4**.

> **Warning:** run this cell once. After it, the late file is already loaded, so a second run adds 0 rows in Run B and the validation fails. To run it again, or if it fails after Run A, first run the Task 2 reset cell, Task 2 and Task 3 again. **Check your work** reads only the table, which still has every row once, so it can pass after a second run: follow the validation cell.

In [0]:
# Task 4: paste the code from the Workshop Guide here

In [0]:
# -- Validation --
assert new_rows_no_files == 0, f"Run A should add 0 rows (checkpoint), but added {new_rows_no_files}"
assert new_rows_late_file == late_file_rows, \
    f"Run B should add exactly the new file's {late_file_rows} rows, but added {new_rows_late_file}"
assert al_count2 == al_count + late_file_rows, "Total rows should be the Task 3 count plus the new file"
print(f"Task 4 OK: Run A +0 rows, Run B +{new_rows_late_file} rows — only the new file was processed")

## Task 5 — Add the Source File and Load Time to Every Row (stretch)

> **Stretch:** optional in class. Do it if you finish early, or after the course. Task 6 does not need it, because it has its own table and checkpoint.

When a bad row shows up in Bronze, you first ask which file it came from and when it was loaded. Add two columns to every row: `_processing_time` with the processing time and `_source_file` with the full path of the source file.

The cell reads the full source folder with its own checkpoint and schema location and writes `orders_with_metadata`. The `writeStream` part is provided. Get the code from the Workshop Guide, **Task 5**.

In [0]:
# Task 5: paste the code from the Workshop Guide here

In [0]:
# -- Validation --
meta_cols = spark.table(metadata_target).columns
assert "_processing_time" in meta_cols, "Missing '_processing_time' — did you use current_timestamp()?"
assert "_source_file" in meta_cols, "Missing '_source_file' — did you use _metadata.file_path?"
print(f"Task 5 OK: Metadata columns added — {meta_cols}")

## Task 6 — Schema Evolution: Keep the Unknown Fields

New fields appear in the files without notice. This Bronze table declares only `order_id` and `customer_id`. Choose the schema evolution mode that keeps every other field as JSON in `_rescued_data`. In this mode no data is lost and the stream does not stop.

The cell reads the full source folder with this partial schema and writes `lab04_orders_rescued`. The `writeStream` part is provided. Get the code from the Workshop Guide, **Task 6**.

In [0]:
# Task 6: paste the code from the Workshop Guide here

In [0]:
# -- Validation --
rescue_cols = spark.table(rescue_target).columns
assert "_rescued_data" in rescue_cols, "Missing '_rescued_data' column — did you set schemaEvolutionMode to 'rescue'?"
rescue_count = spark.table(rescue_target).filter("_rescued_data IS NOT NULL").count()
assert rescue_count > 0, "Expected rescued data for columns not in partial schema"
print(f"Task 6 OK: {rescue_count} rows with rescued data (extra columns captured in _rescued_data)")

## Cleanup

Run this cell only after **Check your work** has passed for every task. It stops the streams, drops the tables that the checks read and deletes the lab folder. In a new session (for example after a break), run the first Setup cell (`%run`) before it.

In [0]:
# Stop any active streams
for s in spark.streams.active:
    s.stop()

# Drop the lab tables: the names need only the first Setup cell (%run), so this cell works in a new session too
for name in ["orders_stream", "lab04_orders_autoloader", "orders_with_metadata", "lab04_orders_rescued"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.{name}")

# Clean checkpoints, schemas and the lab landing zone (all under lab04_work)
dbutils.fs.rm(f"{DATASET_PATH}/lab04_work", True)

print("Lab cleanup complete")

## Lab Complete

In this lab you:
- Loaded files in batch with COPY INTO and ran the load again without duplicates
- Configured Auto Loader (`cloudFiles`) as a streaming source
- Ran the stream with `trigger(availableNow=True)`, so it processed the waiting files and stopped
- Checked that the checkpoint processes each file once: 0 rows with no new files, and only the rows of the new file after it arrived
- Added the metadata columns `_processing_time` and `_source_file` (stretch)
- Kept the fields outside the declared schema in the rescued data column

| Feature | COPY INTO | Auto Loader |
|---------|-----------|-------------|
| API | SQL command | `readStream` / `writeStream` |
| Scalability | Thousands of files | Millions of files |
| Schema evolution | `COPY_OPTIONS ('mergeSchema' = 'true')` | `cloudFiles.schemaEvolutionMode`: `addNewColumns` (default without schema) / `rescue` / `failOnNewColumns` / `none` |
| File tracking | Target table metadata | Checkpoint directory |
| Use case | Simple batch | File-based streaming |

> **Exam Tip:** Auto Loader uses the `cloudFiles` format. COPY INTO is simpler, but Auto Loader scales better: it lists the directory by default, and file notification mode with file events handles millions of files. Stream-static joins and Change Data Feed are in the Bonus notebook.


← <a href="$../demo/04_incremental_ingestion">04 — Incremental Ingestion</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$../../day2/demo/04a_lakeflow_connect_demo">04a — Lakeflow Connect →</a>